## Feature Engineering for Time Series Forecasting

In this notebook we transform the hourly electricity demand data into a supervised learning format by creating time-based and lagged features.


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/electricity_hourly_clean.csv",
    parse_dates=["timestamp"],
    index_col="timestamp"
)

df.head()

,load_mw
timestamp,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


In [3]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["day"] = df.index.day
df["month"] = df.index.month
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

df.head()

,load_mw,hour,dayofweek,day,month,is_weekend
timestamp,,,,,,
2002-01-01 01:00:00,30393.0,1,1,1,1,0
2002-01-01 02:00:00,29265.0,2,1,1,1,0
2002-01-01 03:00:00,28357.0,3,1,1,1,0
2002-01-01 04:00:00,27899.0,4,1,1,1,0
2002-01-01 05:00:00,28057.0,5,1,1,1,0


In [4]:
LAGS = [1, 2, 3, 6, 12, 24, 48, 168]  # hours

for lag in LAGS:
    df[f"lag_{lag}"] = df["load_mw"].shift(lag)

df.head(10)

,load_mw,hour,dayofweek,day,month,is_weekend,lag_1,lag_2,lag_3,lag_6,lag_12,lag_24,lag_48,lag_168
timestamp,,,,,,,,,,,,,,
2002-01-01 01:00:00,30393.0,1,1,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-01-01 02:00:00,29265.0,2,1,1,1,0,30393.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-01-01 03:00:00,28357.0,3,1,1,1,0,29265.0,30393.0,NaN,NaN,NaN,NaN,NaN,NaN
2002-01-01 04:00:00,27899.0,4,1,1,1,0,28357.0,29265.0,30393.0,NaN,NaN,NaN,NaN,NaN
2002-01-01 05:00:00,28057.0,5,1,1,1,0,27899.0,28357.0,29265.0,NaN,NaN,NaN,NaN,NaN
2002-01-01 06:00:00,28654.0,6,1,1,1,0,28057.0,27899.0,28357.0,NaN,NaN,NaN,NaN,NaN
2002-01-01 07:00:00,29308.0,7,1,1,1,0,28654.0,28057.0,27899.0,30393.0,NaN,NaN,NaN,NaN
2002-01-01 08:00:00,29595.0,8,1,1,1,0,29308.0,28654.0,28057.0,29265.0,NaN,NaN,NaN,NaN
2002-01-01 09:00:00,29943.0,9,1,1,1,0,29595.0,29308.0,28654.0,28357.0,NaN,NaN,NaN,NaN


In [5]:
WINDOWS = [3, 6, 12, 24]

for window in WINDOWS:
    df[f"rolling_mean_{window}"] = df["load_mw"].shift(1).rolling(window).mean()
    df[f"rolling_std_{window}"] = df["load_mw"].shift(1).rolling(window).std()

Rows with missing values introduced by lag and rolling features are removed.

In [6]:
df_fe = df.dropna()
assert df_fe.isna().sum().sum() == 0

output_path = "../data/processed/electricity_hourly_features.csv"

df_fe.to_csv(output_path, index=True)

print(f"Saved feature dataset to {output_path}")
print(df_fe.shape)

Saved feature dataset to ../data/processed/electricity_hourly_features.csv
(144414, 22)


In [7]:
split_date = df_fe.index.max() - pd.Timedelta(days=30)

train = df_fe[df_fe.index < split_date]
test = df_fe[df_fe.index >= split_date]

X_train = train.drop(columns=["load_mw"])
y_train = train["load_mw"]

X_test = test.drop(columns=["load_mw"])
y_test = test["load_mw"]

X_train.shape, X_test.shape

((143693, 21), (721, 21))

In [8]:
X_train.to_csv("../data/processed/X_train.csv")
X_test.to_csv("../data/processed/X_test.csv")
y_train.to_csv("../data/processed/y_train.csv")
y_test.to_csv("../data/processed/y_test.csv")